In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
import requests
import time
import os

file = "adblnewshtml.html"

base_path = "../DATA-HTML-STOCK"
html_path = os.path.join(base_path, "webScrapped-htmlfiles", file)
save_path = os.path.join(base_path,"NEPSEDATA",file.replace(".html", "_full_content.csv"))

data = {'Date': [], 'Headline': [], 'Link': [], 'Full Content': []}

with open(html_path, encoding="utf-8") as f:
    soup = BeautifulSoup(f.read(), "html.parser")

rows = soup.find("tbody").find_all("tr")

for row in rows:
    cols = row.find_all("td")
    if len(cols) >= 2:
        date = cols[0].get_text(strip=True)
        headline = cols[1].get_text(strip=True)
        link = cols[1].find("a")["href"].strip()

        try:
            response = requests.get(link, timeout=10)
            response.raise_for_status()
            article_soup = BeautifulSoup(response.content, "html.parser")
            content_div = article_soup.find("div", id="newsdetail-content")
            content = content_div.get_text(separator="\n", strip=True) if content_div else ""
        except:
            content = ""

        data["Date"].append(date)
        data["Headline"].append(headline)
        data["Link"].append(link)
        data["Full Content"].append(content)

        time.sleep(1)

pd.DataFrame(data).to_csv(save_path, index=False)